# Día 13 · Automatización de extremo a extremo

Este notebook conecta retrieval BGE-M3, respuesta RAG, catálogo calibrado privado, validación determinista y perfil agregado. Los archivos privados y checkpoints no se suben a GitHub.

In [ ]:
!pip -q install -r requirements.txt

In [ ]:
import os
from getpass import getpass
from pathlib import Path

if not os.getenv('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('OPENAI_API_KEY: ')

ROOT = Path.cwd()
EMBEDDINGS = ROOT / 'data/processed/embedding/embeddings_bge_m3.parquet'
PRIVATE_CATALOG = Path('/content/private/documentary_items_calibrated.csv')
PRIVATE_CHECKPOINTS = Path('/content/private/checkpoints_day_13')

assert EMBEDDINGS.exists(), EMBEDDINGS
assert PRIVATE_CATALOG.exists(), 'Carga el catálogo privado del Día 5 en /content/private/'

In [ ]:
from src.pipeline.end_to_end import (
    BgeM3Retriever, CalibratedCatalogExtractor, EndToEndPipeline,
    JsonCheckpointStore, OpenAIRagAnswerer, PublishedProfileLoader,
    public_execution_summary,
)

pipeline = EndToEndPipeline(
    retriever=BgeM3Retriever(EMBEDDINGS, top_k=5),
    extractor=CalibratedCatalogExtractor(PRIVATE_CATALOG),
    profiler=PublishedProfileLoader(
        ROOT / 'results/day_11/intelligent_risk_profile.json',
        ROOT / 'results/day_11/category_profile.csv',
    ),
    answerer=OpenAIRagAnswerer(model='gpt-4o-mini'),
    checkpoint_store=JsonCheckpointStore(PRIVATE_CHECKPOINTS),
)

In [ ]:
QUESTION = '¿Qué retrasos o incumplimientos requieren vigilancia y qué evidencia los sustenta?'
result = pipeline.run(QUESTION, request_id='DEMO-DIA-13', resume=True)
public_execution_summary(result)

## Verificación privada

Revisa `result['qa_result']` y `result['risk_result']` únicamente dentro de Colab. No descargues ni publiques el checkpoint completo. El objeto mostrado arriba es el resumen seguro para documentación.